In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# imports
import numpy as np
import pandas as pd
from pathlib import Path
from src.metrics.metrics import organize_multiverse_results, within_bucket_variation

# get all csv files in the data directory for run of interest
data_dir = Path("output/runs/1/data")
csv_files = list(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files.")

# read all files with mixed types
arrays = []
for i, f in enumerate(csv_files):
    arr = np.genfromtxt(f, delimiter=',', dtype=None, encoding='utf-8')
    arrays.append(arr)
    if (i + 1) % 1000 == 0:
        print(f"Loaded {i+1}/{len(csv_files)} files")

# stack all arrays vertically
data = np.concatenate(arrays)
# convert data to pandas df
df = pd.DataFrame(data)
# remove all rows that are exact duplicates, since they are removing the rows of column names
df = df.drop_duplicates()
# make first row the header
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)

In [ ]:
df.head()

In [ ]:
dimension_order = ["spell_gap_threshold",
 "new_job_notif_threshold",
 "new_job_no_notif_threshold",
 "missing_strategy",
 "split_seed",
 "model"]
# prepend mv_dim_ to all dimension names
dimension_order = ["mv_dim_" + dim for dim in dimension_order]
performance_metrics = ["acc", "f1", "roc_auc"]
# prepend test_ to all performance metric names
performance_metrics = ["test_" + metric for metric in performance_metrics]

In [ ]:
# make all cols that start with test_ numeric
for col in df.columns:
    if col.startswith("test_"):
        df[col] = pd.to_numeric(df[col])

In [ ]:
df = organize_multiverse_results(df, dimension_order, performance_metrics)

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
decision_paths = within_bucket_variation(df, "test_acc")

In [ ]:
decision_paths